# Fast Poison-Point Finder (v3)

Dirty-label blended-trigger backdoor on CIFAR-10 with detection and sanitization, evaluated across seeds.

What changed from v2 (each item follows from the v2 seed 0 results):

1. **Fusion no longer includes LOF.** LOF scored below chance (AUC 0.43) because the poison forms a dense cluster. LOF is still reported as a baseline.
2. **Activation clustering orientation is reported.** It flags the smaller of two clusters. When the clusters are close to equal in size that choice is ambiguous, so each trial prints the cluster sizes and an `ambiguous` flag. A label-aware `sep` column (max of AUC and 1 minus AUC) shows how separable the poison is whatever the orientation. `sep` uses ground truth and is a diagnostic, not a defense result.
3. **Sign-aware spectral baseline** (`spectral_sign`): same top singular vector, but it ranks the side with fewer samples instead of squared magnitude. This tests the hypothesis that plain spectral fails at 50 percent poison because the magnitude hides the sign.
4. **Sanitization with spectral and activation clustering plus a random control** at 1.0x the true poison count.
5. **Resumable.** Every trial writes `trial_*.json`, `scores_*.npz` and `ref_*.json` to `RESULTS_DIR`. Rerunning the run cell skips finished work. Mount Drive and point `RESULTS_DIR` there to survive Colab disconnects.
6. Mean and standard deviation across seeds.

Threat model assumption: the defender knows the target class and trained on the poisoned data (embeddings come from the poisoned model).

## 0) Config

In [ ]:
PRESET = "full"   # "quick": seed 0 only. "full": seeds 0, 1 and 2.
PRESETS = {
    "quick": dict(EPOCHS=15, SEEDS=[0]),
    "full":  dict(EPOCHS=15, SEEDS=[0, 1, 2]),
}
CFG = PRESETS[PRESET]

POISON_RATE = 0.01          # fraction of the 50k training set
STRESS_RATE = 0.10          # detection-only stress test (poison is about 50 percent of the target class)
TARGET = 0
TRIGGER_SIZE = 5
ALPHA = 0.6                 # pilot: alpha 0.6 and size 5 at 1 percent gave ASR 0.86 to 0.88. alpha 0.2 did not implant.
BATCH = 256
LR = 0.1
WD = 5e-4

DET_MULTS = [1.0, 2.0]      # budgets (multiples of the true poison count) for the recall columns
SANITIZE_PLAN = [("spectral", 1.0), ("actclust", 1.0), ("random", 1.0)]   # (detector, budget multiple)
ALLOW_WEAK_ATTACK = False   # if False the run stops when the backdoor did not implant

RESULTS_DIR = "results_v3"
# To survive Colab disconnects, mount Drive first and point RESULTS_DIR at it:
# from google.colab import drive; drive.mount("/content/drive"); RESULTS_DIR = "/content/drive/MyDrive/fpf_results_v3"

## 1) Imports and device

In [ ]:
import os, json, time, random, copy
import numpy as np
import torch, torchvision as tv
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import rankdata
from sklearn.neighbors import LocalOutlierFactor
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score, average_precision_score
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == "cuda"
torch.backends.cudnn.benchmark = True
print("torch", torch.__version__, "| device:", device, "| AMP:", USE_AMP)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

## 2) Data (kept on the GPU as float tensors in [0,1])

In [ ]:
MEAN = torch.tensor([0.4914, 0.4822, 0.4465], device=device).view(1, 3, 1, 1)
STD  = torch.tensor([0.2470, 0.2435, 0.2616], device=device).view(1, 3, 1, 1)

def normalize(x):
    return (x - MEAN) / STD

def load_cifar():
    tr = tv.datasets.CIFAR10("./data", train=True, download=True)
    te = tv.datasets.CIFAR10("./data", train=False, download=True)
    f = lambda a: torch.from_numpy(a).permute(0, 3, 1, 2).float().div(255.0)
    return f(tr.data), torch.tensor(tr.targets), f(te.data), torch.tensor(te.targets)

Xtr, ytr, Xte, yte = load_cifar()
Xtr, ytr, Xte, yte = Xtr.to(device), ytr.to(device), Xte.to(device), yte.to(device)
ytr_np = ytr.cpu().numpy()
print(Xtr.shape, Xte.shape)

## 3) Trigger and poisoning

In [ ]:
def trigger_box(h=32, w=32, size=None, margin=1):
    size = TRIGGER_SIZE if size is None else size   # resolved at call time, not definition time
    x1 = w - margin - 1
    y1 = h - margin - 1
    return max(0, x1 - size + 1), max(0, y1 - size + 1), x1, y1

def add_trigger(x, size=None, alpha=None):
    # x: [N,C,H,W] in [0,1]. Blends a white square into the bottom-right corner.
    # alpha=1.0 gives an opaque patch (classic BadNets). Globals are read at call time.
    alpha = ALPHA if alpha is None else alpha
    x0, y0, x1, y1 = trigger_box(size=size)
    out = x.clone()
    patch = out[:, :, y0:y1 + 1, x0:x1 + 1]
    out[:, :, y0:y1 + 1, x0:x1 + 1] = (1 - alpha) * patch + alpha
    return out

def make_poison_indices(labels, rate, target, seed):
    # Dirty-label attack: poison is drawn from NON-target images and relabeled to the target.
    rng = np.random.default_rng(1000 + seed)
    n = int(round(len(labels) * rate))
    cand = np.flatnonzero(labels != target)
    return np.sort(rng.choice(cand, size=n, replace=False))

def build_poisoned(pidx):
    Xp, yp = Xtr.clone(), ytr.clone()
    t = torch.as_tensor(pidx, device=device)
    Xp[t] = add_trigger(Xtr[t])
    yp[t] = TARGET
    return Xp, yp

## 4) Model, augmentation, training

In [ ]:
def make_resnet18():
    m = tv.models.resnet18(weights=None, num_classes=10)
    m.conv1 = nn.Conv2d(3, 64, 3, 1, 1, bias=False)   # CIFAR stem: 3x3 stride 1
    m.maxpool = nn.Identity()                          # no early downsampling
    return m.to(device).to(memory_format=torch.channels_last)

def augment(x):
    # random horizontal flip and random 32x32 crop from a reflect-padded 40x40 image (per sample, on GPU)
    B, C, H, W = x.shape
    flip = torch.rand(B, device=x.device) < 0.5
    x = torch.where(flip[:, None, None, None], x.flip(3), x)
    xp = F.pad(x, (4, 4, 4, 4), mode="reflect")
    i = torch.randint(0, 9, (B,), device=x.device)
    j = torch.randint(0, 9, (B,), device=x.device)
    ar = torch.arange(H, device=x.device)
    rows = (i[:, None] + ar)[:, None, :, None]
    cols = (j[:, None] + ar)[:, None, None, :]
    b = torch.arange(B, device=x.device)[:, None, None, None]
    c = torch.arange(C, device=x.device)[None, :, None, None]
    return xp[b, c, rows, cols]

def train_model(X, y, epochs, seed, tag=""):
    set_seed(seed)
    model = make_resnet18()
    opt = torch.optim.SGD(model.parameters(), lr=LR, momentum=0.9,
                          weight_decay=WD, nesterov=True)
    n = len(X)
    spe = n // BATCH                                   # drop the last partial batch
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=epochs * spe,
                                                pct_start=0.2)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    t0 = time.time()
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(n, device=device)
        loss_sum = torch.zeros((), device=device)
        correct = torch.zeros((), device=device)
        for s in range(spe):
            idx = perm[s * BATCH:(s + 1) * BATCH]
            xb = normalize(augment(X[idx])).contiguous(memory_format=torch.channels_last)
            yb = y[idx]
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
                logits = model(xb)
                loss = F.cross_entropy(logits, yb)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update(); sched.step()
            loss_sum += loss.detach() * len(idx)
            correct += (logits.argmax(1) == yb).sum()
        if (ep + 1) % 5 == 0 or ep == epochs - 1:
            seen = spe * BATCH
            print(f"  [{tag} ep {ep+1}/{epochs}] loss={loss_sum.item()/seen:.4f} "
                  f"train_acc={correct.item()/seen:.4f} ({time.time()-t0:.0f}s)")
    return model

## 5) Evaluation: clean accuracy and ASR (target-class test images excluded)

In [ ]:
@torch.no_grad()
def predict(model, X, bs=1000):
    model.eval()
    out = []
    for i in range(0, len(X), bs):
        xb = normalize(X[i:i + bs]).contiguous(memory_format=torch.channels_last)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            out.append(model(xb).argmax(1))
    return torch.cat(out)

def clean_acc(model):
    return (predict(model, Xte) == yte).float().mean().item()

def attack_asr(model):
    m = yte != TARGET
    return (predict(model, add_trigger(Xte[m])) == TARGET).float().mean().item()

@torch.no_grad()
def embed(model, X, bs=1000):
    # penultimate-layer activations (512-d) of the given model, eval mode, no augmentation
    net = copy.deepcopy(model)
    net.fc = nn.Identity()
    net.eval()
    out = []
    for i in range(0, len(X), bs):
        xb = normalize(X[i:i + bs]).contiguous(memory_format=torch.channels_last)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            out.append(net(xb).float())
    return torch.cat(out).cpu().numpy()

## 6) Detectors (each scores samples INSIDE the target class, higher means more suspicious)

In [ ]:
def mahalanobis_scores(Z, shrink=1e-2):
    mu = Z.mean(0)
    C = np.cov(Z.T)
    d = Z.shape[1]
    C = (1 - shrink) * C + shrink * (np.trace(C) / d) * np.eye(d)
    D = Z - mu
    return np.sqrt(np.einsum("ij,ij->i", D, np.linalg.solve(C, D.T).T))

def spectral_projection(Z):
    # signed projection of the centered class features on the top singular vector
    D = Z - Z.mean(0)
    _, _, Vt = np.linalg.svd(D, full_matrices=False)
    return D @ Vt[0]

def spectral_scores(Z):
    # Tran et al. 2018: squared projection (magnitude only)
    return spectral_projection(Z) ** 2

def spectral_sign_scores(Z):
    # sign-aware variant: rank by position along the top singular vector, oriented to the side with fewer samples
    p = spectral_projection(Z)
    pos, neg = int((p > 0).sum()), int((p < 0).sum())
    if pos > neg:
        p = -p
    return p, {"side_sizes": [min(pos, neg), max(pos, neg)]}

def lof_scores(Z, k=20):
    lof = LocalOutlierFactor(n_neighbors=k, novelty=False)
    lof.fit(Z)
    return -lof.negative_outlier_factor_

def actclust_scores(Z, n_comp=10, seed=0):
    # Simplified activation clustering (Chen et al. 2018): PCA, 2-means, flag the SMALLER cluster
    P = PCA(n_components=min(n_comp, Z.shape[1]), random_state=seed).fit_transform(Z)
    km = KMeans(n_clusters=2, n_init=10, random_state=seed).fit(P)
    sizes = np.bincount(km.labels_, minlength=2)
    small = int(np.argmin(sizes)); big = 1 - small
    d = km.transform(P)
    info = {"cluster_sizes": [int(sizes[small]), int(sizes[big])],   # (flagged, other)
            "ambiguous": bool(sizes.min() / sizes.sum() > 0.35)}     # near-equal clusters: the flagged side is a coin flip
    return d[:, big] - d[:, small], info

def rank_norm(s):
    return rankdata(s) / len(s)

def all_scores(Z):
    ac, ac_info = actclust_scores(Z)
    ss, ss_info = spectral_sign_scores(Z)
    s = {
        "mahalanobis":   mahalanobis_scores(Z),
        "spectral":      spectral_scores(Z),
        "spectral_sign": ss,
        "lof":           lof_scores(Z),
        "actclust":      ac,
    }
    s["fusion"] = (rank_norm(s["mahalanobis"]) + rank_norm(s["spectral"])) / 2   # LOF removed (below chance)
    return s, {"actclust": ac_info, "spectral_sign": ss_info}

def detection_report(scores, isp, n_poison, mults):
    rep = {}
    for name, s in scores.items():
        auc = float(roc_auc_score(isp, s))
        r = {"auc": auc,
             "sep": max(auc, 1.0 - auc),            # label-aware separability, a diagnostic only
             "ap": float(average_precision_score(isp, s)),
             "chance_ap": float(isp.mean())}
        order = np.argsort(-s)
        for m in mults:
            k = int(round(m * n_poison))
            if k >= len(s):                          # budget covers the whole class: recall is meaningless
                r[f"prec@{m}x"] = float("nan"); r[f"recall@{m}x"] = float("nan")
            else:
                tp = int(isp[order[:k]].sum())
                r[f"prec@{m}x"] = tp / k
                r[f"recall@{m}x"] = tp / n_poison
        rep[name] = r
    return rep

## 7) One trial: train, detect, sanitize, retrain (resumable)

In [ ]:
os.makedirs(RESULTS_DIR, exist_ok=True)

def free():
    if device.type == "cuda":
        torch.cuda.empty_cache()

def get_clean_reference(seed):
    path = f"{RESULTS_DIR}/ref_s{seed}.json"
    sig = {"alpha": ALPHA, "size": TRIGGER_SIZE, "epochs": CFG["EPOCHS"],
           "batch": BATCH, "lr": LR, "wd": WD}
    if os.path.exists(path):
        with open(path) as f:
            d = json.load(f)
        if d["sig"] == sig:
            return d["ref"]
    print(f"[seed {seed}] clean reference model")
    m = train_model(Xtr, ytr, CFG["EPOCHS"], seed, "clean")
    ref = {"acc": clean_acc(m), "asr_floor": attack_asr(m)}
    with open(path, "w") as f:
        json.dump({"sig": sig, "ref": ref}, f)
    del m; free()
    return ref

def save_state(state, jpath):
    with open(jpath, "w") as f:
        json.dump(state, f, indent=2, default=float)

def run_trial(seed, rate, plan):
    tag = f"s{seed}_r{rate}"
    jpath = f"{RESULTS_DIR}/trial_{tag}.json"
    npath = f"{RESULTS_DIR}/scores_{tag}.npz"
    sig = {"alpha": ALPHA, "size": TRIGGER_SIZE, "epochs": CFG["EPOCHS"], "rate": rate,
           "target": TARGET, "batch": BATCH, "lr": LR, "wd": WD}

    pidx = make_poison_indices(ytr_np, rate, TARGET, seed)
    n_poison = len(pidx)
    yp_np = ytr_np.copy(); yp_np[pidx] = TARGET
    members = np.flatnonzero(yp_np == TARGET)            # target-class members as the defender sees them
    isp = np.isin(members, pidx)

    state = None
    if os.path.exists(jpath) and os.path.exists(npath):
        with open(jpath) as f:
            cand = json.load(f)
        if cand.get("sig") == sig:
            state = cand
            print(f"[{tag}] resuming: detection done, {len(state['students'])} student(s) done")

    if state is None:
        Xp, yp = build_poisoned(pidx)
        print(f"[seed {seed} rate {rate:.0%} alpha={ALPHA} size={TRIGGER_SIZE}] poison={n_poison} "
              f"| target class size={len(members)} | poison share of class={isp.mean():.1%}")
        pm = train_model(Xp, yp, CFG["EPOCHS"], seed, f"poisoned {rate:.0%}")
        pacc, pasr = clean_acc(pm), attack_asr(pm)
        tp_ = torch.as_tensor(pidx, device=device)
        fit = (predict(pm, Xp[tp_]) == TARGET).float().mean().item()   # poisoned train samples, no augmentation
        print(f"  poisoned model: clean_acc={pacc:.4f} ASR={pasr:.4f} | poison fit on train set={fit:.3f}")
        if pasr < 0.5 and not ALLOW_WEAK_ATTACK:
            raise RuntimeError(
                f"Backdoor did not implant (ASR={pasr:.3f}, poison fit={fit:.3f}). "
                "Pick a working (ALPHA, POISON_RATE, TRIGGER_SIZE) with the pilot cell, set them in the config cell "
                "and re-execute it, or set ALLOW_WEAK_ATTACK=True to continue anyway.")
        ref = get_clean_reference(seed)

        Z = embed(pm, Xp)[members].astype(np.float64)
        scores, info = all_scores(Z)
        det = detection_report(scores, isp, n_poison, DET_MULTS)
        np.savez(npath, members=members, isp=isp, **scores)
        state = {"sig": sig, "seed": seed, "rate": rate, "n_poison": n_poison,
                 "class_size": int(len(members)), "poison_share": float(isp.mean()),
                 "ref": ref, "poisoned": {"acc": pacc, "asr": pasr, "fit": fit},
                 "detection": det, "info": info, "students": []}
        save_state(state, jpath)
        del pm, Xp, yp; free()

    z = np.load(npath)
    scores = {k: z[k] for k in z.files if k not in ("members", "isp")}
    todo = [(n, m) for n, m in plan
            if not any(s["detector"] == n and s["mult"] == m for s in state["students"])]
    if todo:
        Xp, yp = build_poisoned(pidx)
        for name, mult in todo:
            k = int(round(mult * n_poison))
            if name == "random":
                flagged = np.random.default_rng(seed).choice(members, size=k, replace=False)
            else:
                flagged = members[np.argsort(-scores[name])[:k]]
            keep_mask = torch.ones(len(Xp), dtype=torch.bool, device=device)
            keep_mask[torch.as_tensor(flagged, device=device)] = False
            tp = int(np.isin(flagged, pidx).sum())
            print(f"  student ({name}, {mult}x): remove {k} | poison caught={tp}/{n_poison}")
            st = train_model(Xp[keep_mask], yp[keep_mask], CFG["EPOCHS"], seed, f"student {name} {mult}x")
            row = {"detector": name, "mult": mult, "k": k, "poison_removed": tp,
                   "poison_recall": tp / n_poison,
                   "acc": clean_acc(st), "asr": attack_asr(st)}
            print(f"    -> acc={row['acc']:.4f} ASR={row['asr']:.4f}")
            state["students"].append(row)
            save_state(state, jpath)
            del st, keep_mask; free()
        del Xp, yp; free()
    return state

## 7b) Pilot (optional, off by default)

Trains short poisoned models for several (alpha, rate, size) settings and prints clean accuracy, ASR and poison fit.
The v2 pilot already picked alpha 0.6, size 5, rate 1 percent. Set `RUN_PILOT = True` only to re-tune.

In [ ]:
def pilot(configs, epochs=10, seed=0):
    global ALPHA, TRIGGER_SIZE
    keep = (ALPHA, TRIGGER_SIZE)
    rows = []
    for alpha, rate, size in configs:
        ALPHA, TRIGGER_SIZE = alpha, size
        pidx = make_poison_indices(ytr_np, rate, TARGET, seed)
        Xp, yp = build_poisoned(pidx)
        m = train_model(Xp, yp, epochs, seed, f"pilot a={alpha} r={rate} s={size}")
        t = torch.as_tensor(pidx, device=device)
        fit = (predict(m, Xp[t]) == TARGET).float().mean().item()
        rows.append((alpha, rate, size, clean_acc(m), attack_asr(m), fit))
        del m, Xp, yp; free()
    ALPHA, TRIGGER_SIZE = keep
    print(f"{'alpha':>6}{'rate':>7}{'size':>6}{'clean acc':>11}{'ASR':>8}{'train fit':>11}")
    for r in rows:
        print(f"{r[0]:>6}{r[1]:>7}{r[2]:>6}{r[3]:>11.4f}{r[4]:>8.3f}{r[5]:>11.3f}")
    return rows

PILOT_CONFIGS = [   # (alpha, poison rate, trigger size)
    (0.4, 0.01, 5),
    (0.4, 0.02, 5),
    (0.6, 0.01, 5),
    (1.0, 0.01, 5),
    (1.0, 0.01, 3),
]
RUN_PILOT = False   # the pilot already picked alpha 0.6, size 5, rate 1 percent. Set True only to re-tune.
if RUN_PILOT:
    pilot_rows = pilot(PILOT_CONFIGS, epochs=10)

## 8) Run: main experiment (1 percent poison) and the 10 percent stress test

In [ ]:
results = {"config": {"preset": PRESET, **CFG, "poison_rate": POISON_RATE, "stress_rate": STRESS_RATE,
                      "alpha": ALPHA, "trigger_size": TRIGGER_SIZE, "target": TARGET,
                      "sanitize_plan": SANITIZE_PLAN, "det_mults": DET_MULTS},
           "main": [], "stress": None}

for seed in CFG["SEEDS"]:
    results["main"].append(run_trial(seed, POISON_RATE, SANITIZE_PLAN))

# Stress test: poison is about half of the target class. Detection only.
results["stress"] = run_trial(CFG["SEEDS"][0], STRESS_RATE, [])

with open(f"{RESULTS_DIR}/results_v3.json", "w") as f:
    json.dump(results, f, indent=2, default=float)
print(f"saved {RESULTS_DIR}/results_v3.json")

## 9) Tables

In [ ]:
def ms(vals):
    v = np.array(vals, dtype=float)
    n = int(np.sum(~np.isnan(v)))
    if n == 0:
        return float("nan"), 0.0
    return float(np.nanmean(v)), (float(np.nanstd(v, ddof=1)) if n > 1 else 0.0)

def cell(x):
    return f"{x:>9.3f}" if not np.isnan(x) else f"{'n/a':>9}"

def print_detection(trials, title):
    print(title)
    head = (f"{'detector':<14}{'AUC':>14}{'AP':>9}{'sep':>8}{'chanceAP':>10}"
            + "".join(f"{'R@' + str(m) + 'x':>9}" for m in DET_MULTS))
    print(head); print("-" * len(head))
    for name in trials[0]["detection"]:
        g = lambda key: ms([t["detection"][name][key] for t in trials])
        a = g("auc")
        line = f"{name:<14}{a[0]:>8.3f}+-{a[1]:.2f}{g('ap')[0]:>9.3f}{g('sep')[0]:>8.3f}{g('chance_ap')[0]:>10.3f}"
        line += "".join(cell(g(f"recall@{m}x")[0]) for m in DET_MULTS)
        print(line)
    for t in trials:
        ac, ss = t["info"]["actclust"], t["info"]["spectral_sign"]
        print(f"  seed {t['seed']}: actclust cluster sizes (flagged, other) = {ac['cluster_sizes']} "
              f"ambiguous={ac['ambiguous']} | spectral_sign sides (flagged, other) = {ss['side_sizes']}")
    print()

main = results["main"]
print_detection(main, f"Detection inside the target class, poison rate {POISON_RATE:.0%} "
                      f"(mean +- std over {len(main)} seed(s); sep = label-aware separability, a diagnostic)")
print_detection([results["stress"]], f"STRESS: poison rate {STRESS_RATE:.0%} "
                                     f"(poison share of class {results['stress']['poison_share']:.0%})")

b_acc = ms([t["poisoned"]["acc"] for t in main]); b_asr = ms([t["poisoned"]["asr"] for t in main])
fl_acc = ms([t["ref"]["acc"] for t in main]);     fl_asr = ms([t["ref"]["asr_floor"] for t in main])
print("Sanitization (retrained students)")
print(f"clean model     : acc={fl_acc[0]:.4f}  ASR floor={fl_asr[0]:.4f}+-{fl_asr[1]:.3f}")
print(f"poisoned model  : acc={b_acc[0]:.4f}  ASR={b_asr[0]:.4f}+-{b_asr[1]:.3f}")
head = f"{'detector':<10}{'k mult':>7}{'poison out':>12}{'recall':>8}{'acc':>8}{'ASR':>14}{'ASR-floor':>11}"
print(head); print("-" * len(head))
for name, mult in SANITIZE_PLAN:
    rows = [s for t in main for s in t["students"] if s["detector"] == name and s["mult"] == mult]
    po = ms([r["poison_removed"] for r in rows]); rc = ms([r["poison_recall"] for r in rows])
    ac = ms([r["acc"] for r in rows]); asr = ms([r["asr"] for r in rows])
    print(f"{name:<10}{mult:>7}{po[0]:>8.0f}/{main[0]['n_poison']:<3}{rc[0]:>8.3f}{ac[0]:>8.4f}"
          f"{asr[0]:>8.4f}+-{asr[1]:.2f}{asr[0] - fl_asr[0]:>+11.4f}")
print()
print("per seed ASR after sanitization:")
for name, mult in SANITIZE_PLAN:
    vals = [round(s["asr"], 4) for t in main for s in t["students"] if s["detector"] == name and s["mult"] == mult]
    print(f"  {name:<10}{mult}x: {vals}")

## 10) Figures

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.2))

labels, means, stds, pts = [], [], [], []
def add(label, vals):
    labels.append(label); means.append(float(np.mean(vals)))
    stds.append(float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0); pts.append(vals)

add("clean model\n(floor)", [t["ref"]["asr_floor"] for t in main])
add("poisoned\n(before)", [t["poisoned"]["asr"] for t in main])
for name, mult in SANITIZE_PLAN:
    add(f"{name}\n{mult}x", [s["asr"] for t in main for s in t["students"]
                              if s["detector"] == name and s["mult"] == mult])
x = np.arange(len(labels))
a1.bar(x, means, yerr=stds, capsize=4, color="tab:blue", alpha=0.75)
for i, v in enumerate(pts):
    a1.scatter([i] * len(v), v, color="black", s=14, zorder=3)
a1.set_xticks(x); a1.set_xticklabels(labels, fontsize=8); a1.set_ylim(0, 1)
a1.set_ylabel("ASR"); a1.set_title(f"ASR after sanitization ({POISON_RATE:.0%} poison, {len(main)} seed(s))")

names = list(main[0]["detection"].keys())
auc_main = [np.mean([t["detection"][n]["auc"] for t in main]) for n in names]
auc_stress = [results["stress"]["detection"][n]["auc"] for n in names]
sep_stress = [results["stress"]["detection"][n]["sep"] for n in names]
xx = np.arange(len(names))
a2.bar(xx - 0.2, auc_main, 0.4, label=f"AUC at {POISON_RATE:.0%}")
a2.bar(xx + 0.2, auc_stress, 0.4, label=f"AUC at {STRESS_RATE:.0%} (stress)")
a2.scatter(xx + 0.2, sep_stress, marker="_", s=300, color="black", label="separability at stress", zorder=3)
a2.axhline(0.5, linestyle=":", color="black")
a2.set_xticks(xx); a2.set_xticklabels(names, rotation=25, fontsize=8); a2.set_ylim(0, 1.02)
a2.set_ylabel("ROC-AUC inside target class"); a2.set_title("Detection quality"); a2.legend(fontsize=8)

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/v3_summary.png", dpi=150, bbox_inches="tight")
plt.show()

## How to read the output

- **Sanitization is credible only if** ASR after removal approaches the clean-model floor and the random-removal control stays near the pre-sanitization ASR, in every seed.
- **`sep` versus `AUC`**: when they differ a lot the poison is separable but the detector flagged the wrong side. At 50 percent poison, activation clustering and `spectral_sign` pick a side by size, so check the `ambiguous` flag and the printed sizes before reading the stress AUC as detection.
- **Spectral at 50 percent poison**: if `spectral_sign` recovers where plain `spectral` fails, the magnitude-versus-sign explanation is supported. If it does not, the explanation is wrong.
- **LOF** below 0.5 AUC means poison looks denser than clean data. Treat it as a negative result.
- The random control uses the same budget as the detector and draws from the target class only.
- Remaining limitations: the target class is assumed known, embeddings come from the poisoned model, one trigger and one dataset and 15 epochs. Activation clustering is a simplified variant.